In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import numpy as np
from scipy import stats
from scipy.stats import gaussian_kde
import warnings

# Suppress matplotlib legend warnings
warnings.filterwarnings('ignore', message='.*No artists with labels found.*')

# ====================================================================
# 1. FILE PATHS
# ====================================================================
excel_files = {
    '4h': r"4h\SMT_State_Analysis_Test.xlsx",
    '6h': r"6\SMT_State_Analysis_Test.xlsx",
    '8h': r"8\SMT_State_Analysis_Test.xlsx"
}

output_dir = r"output_dir"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# ====================================================================
# 2. DATA LOADING
# ====================================================================
all_data = []
loaded_timepoints = []

print("--- Checking Files ---")
for timepoint, path in excel_files.items():
    if os.path.exists(path):
        df = pd.read_excel(path, sheet_name='Individual_Tracks')
        
        # Validate required columns
        required_cols = {'Type', 'Relative_Position_255'}
        missing = required_cols - set(df.columns)
        if missing:
            print(f"❌ {timepoint} missing columns: {missing}")
            continue
        
        df['Timepoint'] = timepoint
        all_data.append(df)
        loaded_timepoints.append(timepoint)
        print(f"✅ Loaded: {timepoint} ({len(df)} tracks)")
    else:
        print(f"❌ NOT FOUND: {timepoint}")

if len(all_data) < 2:
    print("\nERROR: Not enough data to compare (need at least 2 timepoints)")
    sys.exit(1)

df_total = pd.concat(all_data, ignore_index=True)

# Check for unexpected state types
observed_types = set(df_total['Type'].unique())
expected_types = {'Static', 'Transient', 'Mobile'}
unexpected = observed_types - expected_types
if unexpected:
    print(f"⚠️ Unexpected states found (excluded from analysis): {unexpected}")

print(f"\n📊 Total tracks loaded: {len(df_total)}")

# ====================================================================
# 3. SHARED COLORS & STYLE CONSTANTS
# ====================================================================
TRACK_COLORS = {
    'Static': '#ff7f0e',
    'Transient': '#2ca02c',
    'Mobile': '#1f77b4'
}

# ====================================================================
# 4. STATISTICAL & EFFECT SIZE FUNCTIONS
# ====================================================================
def perform_statistical_tests(data1, data2, label1, label2):
    """
    Performs KS test, Wasserstein distance, Mann-Whitney U test,
    and calculates Effect Sizes (Cohen's d, Cliff's Delta).
    """
    data1_clean = data1[~np.isnan(data1)]
    data2_clean = data2[~np.isnan(data2)]
    
    if len(data1_clean) < 2 or len(data2_clean) < 2:
        return None
    
    n1, n2 = len(data1_clean), len(data2_clean)
    
    # 1. Hypothesis Tests
    ks_stat, ks_pvalue = stats.ks_2samp(data1_clean, data2_clean)
    wasserstein_dist = stats.wasserstein_distance(data1_clean, data2_clean)
    mw_stat, mw_pvalue = stats.mannwhitneyu(data1_clean, data2_clean, alternative='two-sided')
    
    # 2. Effect Sizes
    # Cohen's d (parametric effect size)
    var1, var2 = np.var(data1_clean, ddof=1), np.var(data2_clean, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    cohens_d = (np.mean(data1_clean) - np.mean(data2_clean)) / pooled_std if pooled_std > 0 else 0
    
    # Cliff's Delta (non-parametric effect size)
    cliffs_delta = (2 * mw_stat) / (n1 * n2) - 1
    
    return {
        'Comparison': f'{label1} vs {label2}',
        'N1': n1, 'N2': n2,
        'Mean1_nm': np.mean(data1_clean), 'Mean2_nm': np.mean(data2_clean),
        'Median1_nm': np.median(data1_clean), 'Median2_nm': np.median(data2_clean),
        'KS_statistic': ks_stat, 'KS_pvalue': ks_pvalue,
        'Wasserstein_nm': wasserstein_dist,
        'MW_statistic': mw_stat, 'MW_pvalue': mw_pvalue,
        'Cohens_d': cohens_d, 'Cliffs_delta': cliffs_delta
    }

def format_pvalue(p):
    if p < 0.001: return 'p<0.001***'
    elif p < 0.01: return f'p={p:.3f}**'
    elif p < 0.05: return f'p={p:.3f}*'
    else: return f'p={p:.2f} ns'

# ====================================================================
# 5. PRE-CALCULATE GLOBAL MAX DENSITY FOR ALL KDE PLOTS
# ====================================================================
print("\n--- Calculating Global Y-axis Scale for ALL KDE Plots ---")
global_max_density = 0
x_vals = np.linspace(0, 255, 500)

# 1. Aggregated by Type (for Section 7)
for type_filter in [
    df_total['Type'] == 'Static',
    df_total['Type'] == 'Transient',
    df_total['Type'].isin(['Static', 'Transient']),
    df_total['Type'] == 'Mobile'
]:
    subset_data = df_total[type_filter]['Relative_Position_255'].dropna().values
    if len(subset_data) > 1:
        kde = gaussian_kde(subset_data)
        densities = kde(x_vals)
        global_max_density = max(global_max_density, densities.max())

# 2. Per Timepoint and Type (for Sections 8 & 9)
for timepoint in sorted(df_total['Timepoint'].unique()):
    tp_subset = df_total[df_total['Timepoint'] == timepoint]
    for type_filter in [
        tp_subset['Type'] == 'Static',
        tp_subset['Type'] == 'Transient',
        tp_subset['Type'] == 'Mobile',
        tp_subset['Type'].isin(['Static', 'Transient'])
    ]:
        subset_data = tp_subset[type_filter]['Relative_Position_255'].dropna().values
        if len(subset_data) > 1:
            kde = gaussian_kde(subset_data)
            densities = kde(x_vals)
            global_max_density = max(global_max_density, densities.max())

# Fallback just in case no data was found to prevent Y-axis from being 0
if global_max_density == 0:
    global_max_density = 0.01

print(f"✅ Global max density set to: {global_max_density:.4f}")

# ====================================================================
# 6. STATE PROPORTION ANALYSIS
# ====================================================================
print("\n--- Analyzing State Proportions ---")

# Create contingency table (Timepoint vs State)
contingency = pd.crosstab(df_total['Timepoint'], df_total['Type'])
contingency = contingency.reindex(columns=['Static', 'Transient', 'Mobile'], fill_value=0)

# Overall Chi-square test
chi2_stat, p_chi, dof, expected = stats.chi2_contingency(contingency)
print(f"Overall Chi-square test: χ²={chi2_stat:.3f}, p={p_chi:.4f}")
if (expected < 5).any():
    print("⚠️ Warning: Some expected frequencies < 5. Chi-square may not be valid.")
if p_chi < 0.05:
    print("→ Significant difference in state proportions across timepoints!")

# Calculate percentages for plotting
proportions = contingency.div(contingency.sum(axis=1), axis=0) * 100

# Plot Stacked Bar Chart
fig, ax = plt.subplots(figsize=(8, 6))

# Plot each state separately to have proper legend handles
x_pos = np.arange(len(proportions.index))
width = 0.6

bottom = np.zeros(len(proportions.index))
for state in ['Static', 'Transient', 'Mobile']:
    if state in proportions.columns:
        values = proportions[state].values
        ax.bar(x_pos, values, width, bottom=bottom, label=state,
               color=TRACK_COLORS[state], edgecolor='black', alpha=0.85)
        bottom += values

ax.set_ylabel('Proportion of Tracks (%)', fontsize=12)
ax.set_xlabel('Timepoint', fontsize=12)
ax.set_title('State Distribution Across Timepoints', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(proportions.index, rotation=0)
ax.legend(title='State', loc='upper right', bbox_to_anchor=(1.15, 1.0))

# Add Chi-square annotation - using format_pvalue for consistency
chi_text = f"Chi-square: {format_pvalue(p_chi)}"

ax.text(0.02, 0.95, chi_text, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Add percentage labels inside bars
for container in ax.containers:
    labels = []
    for bar in container:
        h = bar.get_height()
        if h > 5:
            labels.append(f'{h:.0f}%')
        else:
            labels.append('')
    ax.bar_label(container, labels=labels, label_type='center', fontsize=9, color='white', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'State_Proportions.svg'), format='svg', bbox_inches='tight')
plt.close()
print("✅ Saved: State_Proportions.svg")

# Save contingency table to Excel
contingency.to_excel(os.path.join(output_dir, 'State_Proportions_Raw_Counts.xlsx'))

# ====================================================================
# 7. KDE PLOTS WITH STATISTICS & EFFECT SIZES (ORIGINAL PLOTS)
# ====================================================================
plot_configs = [
    {'name': 'Static_Comparison.svg', 'filter': df_total['Type'] == 'Static', 'palette': 'Oranges', 'title': 'Static Tracks', 'prefix': 'Static'},
    {'name': 'Transient_Comparison.svg', 'filter': df_total['Type'] == 'Transient', 'palette': 'mako', 'title': 'Transient Tracks', 'prefix': 'Transient'},
    {'name': 'Total_Bound_Comparison.svg', 'filter': df_total['Type'].isin(['Static', 'Transient']), 'palette': 'magma', 'title': 'Total Bound Population', 'prefix': 'Total_Bound'},
    {'name': 'Mobile_Comparison.svg', 'filter': df_total['Type'] == 'Mobile', 'palette': 'Blues', 'title': 'Mobile Tracks', 'prefix': 'Mobile'}
]

print("\n--- Generating KDE Plots with Stats & Effect Sizes ---")
all_stats_results = []

for config in plot_configs:
    subset = df_total[config['filter']]
    if subset.empty:
        print(f"⚠️  Skipped: {config['name']} (no data)")
        continue
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # First plot: fill with color
    sns.kdeplot(data=subset, x='Relative_Position_255', hue='Timepoint',
                palette=config['palette'], fill=True, common_norm=False, alpha=0.3, ax=ax)
    
    # Second plot: black outline only (no fill)
    sns.kdeplot(data=subset, x='Relative_Position_255', hue='Timepoint',
                palette=config['palette'], fill=False, common_norm=False, ax=ax,
                linewidth=1.5, color='black')
    
    ax.set_title(config['title'], fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Normalized distance from Membrane (AU)', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    
    # X-axis from 0 to 255
    ax.set_xlim(0, 255)
    
    # Y-axis same across ALL KDE graphs
    ax.set_ylim(0, global_max_density * 1.1)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    
    # Only add legend if there are labeled artists
    handles, labels = ax.get_legend_handles_labels()
    if labels:
        ax.legend(title='Timepoint', loc='upper right', bbox_to_anchor=(1.0, 1.0))
    
    # Sample size
    ax.text(0.02, 0.98, f"n = {len(subset)} tracks", transform=ax.transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Pairwise Stats & Effect Sizes
    stats_text_lines = []
    
    timepoints = sorted(subset['Timepoint'].unique())
    for i in range(len(timepoints)):
        for j in range(i + 1, len(timepoints)):
            tp1, tp2 = timepoints[i], timepoints[j]
            data1 = subset[subset['Timepoint'] == tp1]['Relative_Position_255'].values
            data2 = subset[subset['Timepoint'] == tp2]['Relative_Position_255'].values
            
            res = perform_statistical_tests(data1, data2, tp1, tp2)
            if res:
                res['Plot_Type'] = config['prefix']
                all_stats_results.append(res)
                
                # Compact formatting for the plot text box
                stats_text_lines.append(f"{tp1} vs {tp2}:")
                stats_text_lines.append(f"  KS: {format_pvalue(res['KS_pvalue'])} | MW: {format_pvalue(res['MW_pvalue'])}")
                stats_text_lines.append(f"  W: {res['Wasserstein_nm']:.1f} AU | d: {res['Cohens_d']:.2f} | Δ: {res['Cliffs_delta']:.2f}")
                stats_text_lines.append("")
    
    if stats_text_lines:
        stats_text = "\n".join(stats_text_lines[:-1])
        ax.text(0.98, 0.98, stats_text, transform=ax.transAxes, fontsize=8.5,
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.85), family='monospace')
    
    plt.savefig(os.path.join(output_dir, config['name']), format='svg', bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {config['name']} (n={len(subset)})")

# ====================================================================
# 8. NEW: KDE PLOTS PER TIMEPOINT (ALL TRACK TYPES TOGETHER)
# ====================================================================
print("\n--- Generating Per-Timepoint KDE Plots (All Track Types) ---")

for timepoint in sorted(df_total['Timepoint'].unique()):
    tp_subset = df_total[df_total['Timepoint'] == timepoint]
    
    if tp_subset.empty:
        print(f"⚠️  Skipped: {timepoint}_TrackTypes.svg (no data)")
        continue
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot each track type
    track_types = ['Static', 'Transient', 'Mobile']
    
    for track_type in track_types:
        type_data = tp_subset[tp_subset['Type'] == track_type]
        if not type_data.empty:
            # First: colored fill
            sns.kdeplot(data=type_data, x='Relative_Position_255',
                       fill=True, alpha=0.3, ax=ax,
                       color=TRACK_COLORS[track_type], label=track_type)
            # Second: black outline
            sns.kdeplot(data=type_data, x='Relative_Position_255',
                       fill=False, ax=ax,
                       color='black', linewidth=1.5)
    
    ax.set_title(f'{timepoint} - All Track Types', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Normalized distance from Membrane (AU)', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    
    # X-axis from 0 to 255
    ax.set_xlim(0, 255)
    
    # Y-axis same across ALL KDE graphs
    ax.set_ylim(0, global_max_density * 1.1)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    
    # Legend
    ax.legend(title='Track Type', loc='upper right', bbox_to_anchor=(1.0, 1.0))
    
    # Sample size per track type
    sample_text = f"n = {len(tp_subset)} tracks\n"
    for track_type in track_types:
        n_type = len(tp_subset[tp_subset['Type'] == track_type])
        sample_text += f"  {track_type}: {n_type}\n"
    
    ax.text(0.02, 0.98, sample_text.strip(), transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Pairwise Stats between track types
    stats_text_lines = []
    
    for i in range(len(track_types)):
        for j in range(i + 1, len(track_types)):
            type1, type2 = track_types[i], track_types[j]
            data1 = tp_subset[tp_subset['Type'] == type1]['Relative_Position_255'].values
            data2 = tp_subset[tp_subset['Type'] == type2]['Relative_Position_255'].values
            
            res = perform_statistical_tests(data1, data2, type1, type2)
            if res:
                res['Plot_Type'] = f'{timepoint}_TrackTypes'
                all_stats_results.append(res)
                
                stats_text_lines.append(f"{type1} vs {type2}:")
                stats_text_lines.append(f"  KS: {format_pvalue(res['KS_pvalue'])} | MW: {format_pvalue(res['MW_pvalue'])}")
                stats_text_lines.append(f"  W: {res['Wasserstein_nm']:.1f} AU | d: {res['Cohens_d']:.2f} | Δ: {res['Cliffs_delta']:.2f}")
                stats_text_lines.append("")
    
    if stats_text_lines:
        stats_text = "\n".join(stats_text_lines[:-1])
        ax.text(0.98, 0.98, stats_text, transform=ax.transAxes, fontsize=8.5,
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.85), family='monospace')
    
    plt.savefig(os.path.join(output_dir, f'{timepoint}_TrackTypes.svg'), format='svg', bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {timepoint}_TrackTypes.svg (n={len(tp_subset)})")

# ====================================================================
# 9. MOBILE vs TOTAL BOUND KDE PLOTS PER TIMEPOINT
# ====================================================================
print("\n--- Generating Mobile vs Total Bound KDE Plots per Timepoint ---")

for timepoint in sorted(df_total['Timepoint'].unique()):
    tp_subset = df_total[df_total['Timepoint'] == timepoint]
    
    # Filter data for Mobile and Total Bound (Static + Transient)
    mobile_subset = tp_subset[tp_subset['Type'] == 'Mobile']
    bound_subset = tp_subset[tp_subset['Type'].isin(['Static', 'Transient'])]
    
    if mobile_subset.empty or bound_subset.empty:
        print(f"⚠️  Skipped: {timepoint}_Mobile_vs_Bound.svg (insufficient data)")
        continue
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot Mobile - colored fill
    sns.kdeplot(data=mobile_subset, x='Relative_Position_255',
                fill=True, alpha=0.3, ax=ax,
                color=TRACK_COLORS['Mobile'], label='Mobile')
    # Plot Mobile - black outline
    sns.kdeplot(data=mobile_subset, x='Relative_Position_255',
                fill=False, ax=ax,
                color='black', linewidth=1.5)
    
    # Plot Total Bound - colored fill
    sns.kdeplot(data=bound_subset, x='Relative_Position_255',
                fill=True, alpha=0.3, ax=ax,
                color=TRACK_COLORS['Static'], label='Total Bound (Static + Transient)')
    # Plot Total Bound - black outline
    sns.kdeplot(data=bound_subset, x='Relative_Position_255',
                fill=False, ax=ax,
                color='black', linewidth=1.5)
    
    ax.set_title(f'{timepoint} - Mobile vs Total Bound', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Normalized distance from Membrane (AU)', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    
    # X-axis from 0 to 255
    ax.set_xlim(0, 255)
    
    # Y-axis same across ALL KDE graphs (using pre-calculated global max)
    ax.set_ylim(0, global_max_density * 1.1)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    
    # Legend (Moved slightly outside so it doesn't overlap the stats box)
    ax.legend(title='Population', loc='upper left', bbox_to_anchor=(1.02, 1.0))
    
    # Sample size text box
    sample_text = f"n = {len(tp_subset)} tracks\n"
    sample_text += f"  Mobile: {len(mobile_subset)}\n"
    sample_text += f"  Total Bound: {len(bound_subset)}"
    
    ax.text(0.02, 0.98, sample_text, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Statistical comparison between Mobile and Total Bound
    data1 = mobile_subset['Relative_Position_255'].values
    data2 = bound_subset['Relative_Position_255'].values
    
    res = perform_statistical_tests(data1, data2, 'Mobile', 'Total Bound')
    if res:
        # Add to master list so it gets exported to Excel later
        res['Plot_Type'] = f'{timepoint}_Mobile_vs_Bound'
        all_stats_results.append(res)
        
        stats_text_lines = []
        stats_text_lines.append(f"Mobile vs Total Bound:")
        stats_text_lines.append(f"  KS: {format_pvalue(res['KS_pvalue'])} | MW: {format_pvalue(res['MW_pvalue'])}")
        stats_text_lines.append(f"  W: {res['Wasserstein_nm']:.1f} AU | d: {res['Cohens_d']:.2f} | Δ: {res['Cliffs_delta']:.2f}")
        
        stats_text = "\n".join(stats_text_lines)
        ax.text(0.98, 0.98, stats_text, transform=ax.transAxes, fontsize=8.5,
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.85), family='monospace')
    
    plt.savefig(os.path.join(output_dir, f'{timepoint}_Mobile_vs_Bound.svg'), format='svg', bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {timepoint}_Mobile_vs_Bound.svg (n={len(tp_subset)})")

# ====================================================================
# 10. EXPORT STATISTICAL SUMMARY TO EXCEL
# ====================================================================
if all_stats_results:
    stats_df = pd.DataFrame(all_stats_results)
    
    column_order = [
        'Plot_Type', 'Comparison', 'N1', 'N2',
        'Mean1_nm', 'Mean2_nm', 'Median1_nm', 'Median2_nm',
        'KS_statistic', 'KS_pvalue', 'Wasserstein_nm',
        'MW_statistic', 'MW_pvalue',
        'Cohens_d', 'Cliffs_delta'
    ]
    stats_df = stats_df[column_order]
    
    stats_output_path = os.path.join(output_dir, 'Statistical_Comparisons_Summary.xlsx')
    
    with pd.ExcelWriter(stats_output_path) as writer:
        stats_df.to_excel(writer, sheet_name='Distance_Stats', index=False)
        contingency.to_excel(writer, sheet_name='State_Proportions')
        
    print(f"\n📊 Statistical summary saved: {stats_output_path}")
    print(f"   Total comparisons performed: {len(stats_df)}")

print(f"\n--- SUCCESS ---")
print(f"All SVG figures and Excel reports saved in: {output_dir}")